# Fast No RL 05 - Final Export and Local Checks

Run only after the selected no-RL evaluation is complete. Classical export contains no trained model and requires no training. Supervised export requires only supervised training, not self-play. No upload happens automatically.

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project Setup

Uses `configs/fast_no_rl.yaml`. No league, self-play, PPO, or DQN is run. Colab's installed PyTorch is preserved.

In [ ]:
from pathlib import Path
import sys
import subprocess
from IPython.display import display
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

def show_figure(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl/non_rl.py").is_file():
    raise FileNotFoundError(f"Place the updated project contents directly in {PROJECT_ROOT}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                       str(PROJECT_ROOT / "requirements_colab.txt")])
sys.path.insert(0, str(PROJECT_ROOT))
from chess_rl.non_rl import load_no_rl_config
from chess_rl.reproducibility import read_json, sha256
cfg = load_no_rl_config(PROJECT_ROOT, "fast_no_rl.yaml")
print("Run:", cfg["run_id"], "| Variant:", cfg["non_rl"]["variant"])
print("Root:", PROJECT_ROOT)


## Verify the Selected Run

Classical runtime uses only standard library and python-chess. Supervised runtime adds CPU PyTorch with one thread. It uses the original encoder/model/search code, not a separately rewritten implementation.

In [ ]:
from chess_rl.non_rl import selected_no_rl
selection = selected_no_rl(PROJECT_ROOT, cfg["run_id"])
print("Variant:", selection["variant"])
print("Training:", selection["training_status"])
print("Evaluation complete:", selection["evaluation_complete"])

## Export and Check

This is the only no-RL notebook that builds/audits the submission archive or performs fresh-process deployment checks. It checks legal moves, low clocks, import dependencies, archive root/size, and final live games. Supervised weights also undergo a round trip and missing-weight fallback check. Refer to the current official rules before uploading.

In [ ]:
from chess_rl.non_rl_export import final_no_rl_checks
report = final_no_rl_checks(PROJECT_ROOT, cfg["run_id"])
print("Local status:", report["status"])
print("Submission:", report["submission"])
print("Uncompressed bytes:", report["uncompressed_bytes"])
print("Platform acceptance:", report["platform_acceptance"])

## Submission Location

The ZIP has agent.py at its root. Classical coefficients are in config.json, with no neural weight file. For supervised, it additionally contains team-trained CPU weights. Do not upload the portable project ZIP as your agent. Actual server acceptance remains authoritative.

In [ ]:
print(PROJECT_ROOT / "exports/no_rl" / cfg["run_id"] / cfg["non_rl"]["variant"] / "submission.zip")
print("Official documentation: https://aichessathon.com/docs")